# Machine Learning Training (Tree-Based Models) - 42-Day Horizon
This notebook trains 4 tree-based models across 6 commodities for a **42-Day horizon**:
1. **XGBoost**
2. **LightGBM**
3. **CatBoost**
4. **Random Forest**

Target variables are dynamically shifted by 42 days. Anomaly clipping is set to [-80%, +150%]. Overfitting protections are increased to handle overlapping targets.

In [1]:
import pandas as pd
import numpy as np
import os
import json
import warnings
import joblib
import plotly.express as px
import plotly.graph_objects as go
from tqdm.notebook import tqdm
from IPython.display import display, HTML, clear_output
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

warnings.filterwarnings('ignore')

# CONFIGURATION
HORIZON = 42

data_dir = './data'
results_dir = './results'
models_dir = './models'
os.makedirs(results_dir, exist_ok=True)
os.makedirs(models_dir, exist_ok=True)
commodities = ['gold', 'silver', 'copper', 'natural_gas', 'crude_oil', 'wheat']


In [2]:
datasets = {}
print("Loading and validating datasets...")
for name in commodities:
    file_path = os.path.join(data_dir, f"{name}.csv")
    if os.path.exists(file_path):
        df = pd.read_csv(file_path, index_col='Date', parse_dates=True)
        datasets[name] = df
        print(f"✅ {name.upper()}: {df.shape} | Dates: {df.index.min().date()} to {df.index.max().date()} | NaNs: {df.isna().sum().sum()}")
    else:
        print(f"❌ {name.upper()}: Not found")


Loading and validating datasets...
✅ GOLD: (3174, 36) | Dates: 2014-06-02 to 2026-07-30 | NaNs: 0
✅ SILVER: (3174, 35) | Dates: 2014-06-02 to 2026-07-30 | NaNs: 0
✅ COPPER: (2914, 34) | Dates: 2015-06-01 to 2026-07-30 | NaNs: 0
✅ NATURAL_GAS: (3183, 36) | Dates: 2014-05-20 to 2026-07-30 | NaNs: 0
✅ CRUDE_OIL: (2587, 35) | Dates: 2016-08-31 to 2026-07-30 | NaNs: 0
✅ WHEAT: (3174, 37) | Dates: 2014-06-02 to 2026-07-30 | NaNs: 0


### Train / Validation / Test Split Strategy
- **Train (70%)**: Used to fit the trees.
- **Validation (10%)**: Used for early stopping (preventing overfitting).
- **Test (20%)**: Completely unseen data used for final metrics.


In [3]:
def calculate_metrics(y_true, y_pred, y_true_prev):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    
    # MAPE handling zeros
    nonzero = y_true != 0
    mape = np.mean(np.abs((y_true[nonzero] - y_pred[nonzero]) / y_true[nonzero])) * 100
    
    # R2 Score
    r2 = r2_score(y_true, y_pred)
    
    # Directional Accuracy
    actual_dir = np.sign(y_true - y_true_prev)
    pred_dir = np.sign(y_pred - y_true_prev)
    correct_dir = (actual_dir == pred_dir)
    dir_acc = np.mean(correct_dir) * 100
    
    # Naive Baseline (predict today's price for horizon)
    naive_mae = mean_absolute_error(y_true, y_true_prev)
    
    # Improvement vs Naive
    imp_pct = ((naive_mae - mae) / naive_mae) * 100 if naive_mae > 0 else 0
    
    return {
        'MAE': round(mae, 4),
        'RMSE': round(rmse, 4),
        'MAPE': round(mape, 2),
        'Dir_Acc': round(dir_acc, 2),
        'R2': round(r2, 4),
        'Naive_MAE': round(naive_mae, 4),
        'Improvement_Pct': round(imp_pct, 2)
    }

def get_train_val_test(df, horizon):
    drop_cols = ['Target_Close_Next', 'Target_Return_Next', 'Target_Direction', 
                 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
    features = [col for col in df.columns if col not in drop_cols]
    
    df_copy = df.copy()
    
    # Dynamic 42-day shifting
    df_copy[f'Target_Close_{horizon}d'] = df_copy['Close'].shift(-horizon)
    df_copy[f'Target_Return_{horizon}d'] = ((df_copy[f'Target_Close_{horizon}d'] - df_copy['Close']) / df_copy['Close']).clip(lower=-0.8, upper=1.5)
    
    # Drop NaNs at the end caused by shifting
    df_copy = df_copy.dropna(subset=[f'Target_Return_{horizon}d'])
    
    X = df_copy[features]
    y_return = df_copy[f'Target_Return_{horizon}d']
    
    total_len = len(df_copy)
    train_end = int(total_len * 0.7)
    val_end = int(total_len * 0.8)
    
    X_train = X.iloc[:train_end]
    X_val = X.iloc[train_end:val_end]
    X_test = X.iloc[val_end:]
    
    y_train = y_return.iloc[:train_end]
    y_val = y_return.iloc[train_end:val_end]
    y_test = y_return.iloc[val_end:]
    
    y_true_price_test = df_copy[f'Target_Close_{horizon}d'].iloc[val_end:].values
    y_true_prev_test = df_copy['Close'].iloc[val_end:].values
    
    return X_train, X_val, X_test, y_train, y_val, y_test, y_true_price_test, y_true_prev_test, features


In [4]:
results = {}
feature_importances = {}

for name, df in tqdm(datasets.items(), desc="Commodities"):
    X_train, X_val, X_test, y_train, y_val, y_test, y_true_price, y_true_prev, features = get_train_val_test(df, HORIZON)
    
    eval_set_xgb_lgb = [(X_val, y_val)]
    
    print(f"\n{'='*50}\nTraining {name.upper()} ({HORIZON}-Day Horizon)...\n{'='*50}")
    
    res = {}
    fi = {}
    
    # 1. XGBoost (Lowered LR for overlapping target generalization)
    xgb = XGBRegressor(n_estimators=200, learning_rate=0.03, max_depth=6, subsample=0.8, 
                       colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0, random_state=42,
                       early_stopping_rounds=20)
    xgb.fit(X_train, y_train, eval_set=eval_set_xgb_lgb, verbose=False)
    xgb_ret = xgb.predict(X_test)
    xgb_price = y_true_prev * (1 + xgb_ret)
    res['XGBoost'] = calculate_metrics(y_true_price, xgb_price, y_true_prev)
    xgb.save_model(os.path.join(models_dir, f'{name}_xgboost_{HORIZON}d.json'))
    fi['XGBoost'] = xgb.feature_importances_
    
    # 2. LightGBM (Lowered LR)
    lgb = LGBMRegressor(n_estimators=200, learning_rate=0.03, max_depth=6, subsample=0.8, 
                        colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0, random_state=42, verbose=-1)
    from lightgbm import early_stopping
    lgb.fit(X_train, y_train, eval_set=eval_set_xgb_lgb, callbacks=[early_stopping(stopping_rounds=20, verbose=False)])
    lgb_ret = lgb.predict(X_test)
    lgb_price = y_true_prev * (1 + lgb_ret)
    res['LightGBM'] = calculate_metrics(y_true_price, lgb_price, y_true_prev)
    lgb.booster_.save_model(os.path.join(models_dir, f'{name}_lightgbm_{HORIZON}d.txt'))
    fi['LightGBM'] = lgb.feature_importances_
    
    # 3. CatBoost (Lowered LR)
    cat = CatBoostRegressor(iterations=200, learning_rate=0.03, depth=6, l2_leaf_reg=3.0, 
                            random_seed=42, verbose=0, early_stopping_rounds=20)
    cat.fit(X_train, y_train, eval_set=(X_val, y_val))
    cat_ret = cat.predict(X_test)
    cat_price = y_true_prev * (1 + cat_ret)
    res['CatBoost'] = calculate_metrics(y_true_price, cat_price, y_true_prev)
    cat.save_model(os.path.join(models_dir, f'{name}_catboost_{HORIZON}d.cbm'))
    fi['CatBoost'] = cat.feature_importances_
    
    # 4. Random Forest (Increased min_samples_leaf to 10 for overlapping targets)
    rf = RandomForestRegressor(n_estimators=300, max_depth=12, min_samples_leaf=10, 
                               max_features='sqrt', random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    rf_ret = rf.predict(X_test)
    rf_price = y_true_prev * (1 + rf_ret)
    res['RandomForest'] = calculate_metrics(y_true_price, rf_price, y_true_prev)
    joblib.dump(rf, os.path.join(models_dir, f'{name}_randomforest_{HORIZON}d.pkl'))
    fi['RandomForest'] = rf.feature_importances_
    
    results[name] = res
    feature_importances[name] = pd.DataFrame(fi, index=features)
    
    # Display running results
    df_res = pd.DataFrame(res).T
    display(df_res)


Commodities:   0%|          | 0/6 [00:00<?, ?it/s]


Training GOLD (42-Day Horizon)...


,MAE,RMSE,MAPE,Dir_Acc,R2,Naive_MAE,Improvement_Pct
XGBoost,265.0718,340.3123,7.27,76.24,0.8519,273.0668,2.93
LightGBM,264.0314,340.1997,7.23,76.24,0.8520,273.0668,3.31
CatBoost,264.3805,340.4998,7.24,76.24,0.8518,273.0668,3.18
RandomForest,288.7518,358.8323,8.07,32.54,0.8354,273.0668,-5.74



Training SILVER (42-Day Horizon)...


,MAE,RMSE,MAPE,Dir_Acc,R2,Naive_MAE,Improvement_Pct
XGBoost,6.5964,11.0568,12.09,70.65,0.6862,6.8733,4.03
LightGBM,6.7327,11.1570,12.42,70.65,0.6805,6.8733,2.05
CatBoost,6.5913,11.1407,12.01,71.29,0.6814,6.8733,4.10
RandomForest,6.6161,11.3347,12.02,70.65,0.6703,6.8733,3.74


...

,MAE,RMSE,MAPE,Dir_Acc,R2,Naive_MAE,Improvement_Pct
XGBoost,0.7612,0.9359,25.80,55.17,-0.4747,0.6275,-21.30
LightGBM,0.6704,0.8508,22.57,54.53,-0.2187,0.6275,-6.83
CatBoost,0.7299,0.9143,24.22,44.83,-0.4076,0.6275,-16.32
RandomForest,0.7233,0.8946,24.24,52.62,-0.3476,0.6275,-15.26



Training CRUDE_OIL (42-Day Horizon)...


,MAE,RMSE,MAPE,Dir_Acc,R2,Naive_MAE,Improvement_Pct
XGBoost,9.5806,13.4254,12.80,39.69,-0.2479,8.6952,-10.18
LightGBM,9.6550,13.4934,12.91,39.69,-0.2606,8.6952,-11.04
CatBoost,9.5068,13.3288,12.70,39.69,-0.2300,8.6952,-9.33
RandomForest,10.0840,14.4367,13.38,44.99,-0.4430,8.6952,-15.97



Training WHEAT (42-Day Horizon)...


,MAE,RMSE,MAPE,Dir_Acc,R2,Naive_MAE,Improvement_Pct
XGBoost,36.7176,47.6372,6.41,58.69,-0.1996,39.2061,6.35
LightGBM,34.5054,44.8571,6.01,59.49,-0.0636,39.2061,11.99
CatBoost,39.2511,51.2380,6.93,50.08,-0.3878,39.2061,-0.11
RandomForest,31.7720,43.2516,5.51,61.08,0.0111,39.2061,18.96


In [5]:
out_path = os.path.join(results_dir, f'stage2_ml_metrics_{HORIZON}d.json')
with open(out_path, 'w') as f:
    json.dump(results, f, indent=4)
print(f"Metrics saved to {out_path}")


Metrics saved to ./results/stage2_ml_metrics_42d.json


In [6]:
from plotly.subplots import make_subplots

for name in commodities:
    df_fi = feature_importances[name]
    # Normalize each column to 100%
    df_fi_norm = df_fi.div(df_fi.sum(axis=0), axis=1) * 100
    
    fig = make_subplots(rows=2, cols=2, subplot_titles=("XGBoost", "LightGBM", "CatBoost", "Random Forest"))
    models = ["XGBoost", "LightGBM", "CatBoost", "RandomForest"]
    
    for i, model_name in enumerate(models):
        row = (i // 2) + 1
        col = (i % 2) + 1
        
        top_10 = df_fi_norm[model_name].sort_values(ascending=True).tail(10)
        
        fig.add_trace(go.Bar(x=top_10.values, y=top_10.index, orientation='h', name=model_name), row=row, col=col)
        
    fig.update_layout(height=700, title_text=f"{name.upper()} ({HORIZON}d): Top 10 Features Broken Down by Model", showlegend=False)
    fig.show()
